# 🤖 RAG Chatbot — Gemini 2.5 Flash + BM25 Hybrid Retrieval
**Hệ thống hỏi đáp PDF** · Dense + BM25 + RRF Hybrid · Evaluation Metrics · Gradio UI

## 📦 BƯỚC 1: Cài đặt thư viện
Chạy ô này, sau đó **Runtime → Restart Session**, rồi mới chạy Bước 2.

In [ ]:
!pip install -q langchain langchain-community langchain-google-genai langchain-chroma langchain-text-splitters
!pip install -q pypdf gradio rank_bm25 pysqlite3-binary
print(' CÀI ĐẶT HOÀN TẤT! Hãy vào Runtime → Restart Session trước khi chạy Bước 2.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 k

## ⚙️ BƯỚC 2: Khởi tạo hệ thống RAG
Lấy Gemini API Key miễn phí tại: https://aistudio.google.com/app/apikey

Upload file PDF lên Colab (panel trái → biểu tượng 📁 → Upload) trước khi chạy.

In [ ]:
import os, sys, warnings, time, hashlib
warnings.filterwarnings('ignore')

# ── 1. API Key ────────────────────────────────────────────────────────────────
from getpass import getpass
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass('🔑 Nhập Gemini API Key: ')

# ── 2. Vá lỗi SQLite (giữ lại phòng khi môi trường vẫn có pysqlite3) ─────────
try:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')
except ImportError:
    pass

# ── 3. Import thư viện ────────────────────────────────────────────────────────
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI          # chỉ giữ LLM
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from pydantic import Field, PrivateAttr                            # FIX: thêm PrivateAttr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from typing import List

# ── 4. Nạp PDF ────────────────────────────────────────────────────────────────
print('📂 Đang nạp PDF từ /content/ ...')
loader = DirectoryLoader('/content/', glob='./*.pdf', loader_cls=PyPDFLoader, show_progress=True)
raw_data = loader.load()

if not raw_data:
    print('⚠️  Không tìm thấy file PDF. Hãy upload trước!')
else:
    print(f'✅ Đã nạp {len(raw_data)} trang từ {len(set(d.metadata["source"] for d in raw_data))} file.')

    # ── 5. Tiền xử lý ────────────────────────────────────────────────────────
    data = []
    for doc in raw_data:
        page_num = doc.metadata.get('page', 0)
        if 0 <= page_num < 50:
            clean_lines = [line.strip() for line in doc.page_content.splitlines() if line.strip()]
            doc.page_content = '\n'.join(clean_lines)
            data.append(doc)
    print(f'🧹 Sau lọc trang 1–50: còn {len(data)} trang.')

    # ── 6. Chunking ───────────────────────────────────────────────────────────
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = splitter.split_documents(data)
    for c in chunks:
        c.metadata['source'] = os.path.basename(c.metadata.get('source', 'unknown'))
        c.metadata['page']   = c.metadata.get('page', 0) + 1
    print(f'✂️  Đã chia thành {len(chunks)} chunks (chunk_size=1000, overlap=100).')
        # ── 6.5. Tự động trích xuất viết tắt từ corpus ───────────────────────────
    import re

    def build_abbrev_map(chunks):
        abbrev_map = {}
        p1 = re.compile(
            r'([A-Z][a-zA-Z]+(?:[\s\n]+[A-Z][a-zA-Z]+){1,5})'
            r'[\s\n]*\(([A-Z]{2,8})\)',
            re.MULTILINE
        )
        p2 = re.compile(
            r'([A-Z]{2,}(?:[\s\n]+[A-Z]{2,}){1,5})'
            r'[\s\n]*\(([A-Z]{2,8})\)',
            re.MULTILINE
        )
        p3 = re.compile(
            r'\b([A-Z]{2,8})\b'
            r'[\s\n]*\(([A-Z][a-zA-Z]+(?:[\s\n]+[A-Z][a-zA-Z]+){1,5})\)',
            re.MULTILINE
        )
        for chunk in chunks:
            text = chunk.page_content
            for match in p1.finditer(text):
                abbrev = match.group(2).lower().strip()
                full   = match.group(1).lower().strip().replace('\n', ' ')
                abbrev_map[abbrev] = full
            for match in p2.finditer(text):
                abbrev = match.group(2).lower().strip()
                full   = match.group(1).lower().strip().replace('\n', ' ')
                if abbrev not in abbrev_map:
                    abbrev_map[abbrev] = full
            for match in p3.finditer(text):
                abbrev = match.group(1).lower().strip()
                full   = match.group(2).lower().strip().replace('\n', ' ')
                if abbrev not in abbrev_map:
                    abbrev_map[abbrev] = full
        return abbrev_map

    def expand_query(query: str) -> str:
        """Thay viết tắt trong query bằng dạng đầy đủ trước khi truyền vào retriever."""
        tokens = query.lower().split()
        return " ".join([ABBREV_MAP.get(tok, tok) for tok in tokens])

    ABBREV_MAP = build_abbrev_map(chunks)
    print(f"✅ Tìm được {len(ABBREV_MAP)} viết tắt:")
    for k, v in ABBREV_MAP.items():
        print(f"   '{k}' → '{v}'")

    # ── 7. TF-IDF Retriever (thay thế Gemini Embedding + Chroma) ─────────────
    class TfidfRetriever(BaseRetriever):
        """
        Custom LangChain-compatible retriever dùng TF-IDF + Cosine Similarity.
        Kế thừa BaseRetriever → .invoke(query) hoạt động chuẩn LangChain.
        """
        chunks: List[Document] = Field(description="Danh sách Document sau khi chunk")
        k: int                 = Field(default=5, description="Số document trả về")

        # FIX: dùng PrivateAttr() thay vì gán = None trực tiếp
        _vectorizer:   TfidfVectorizer = PrivateAttr(default=None)
        _tfidf_matrix: np.ndarray      = PrivateAttr(default=None)

        def model_post_init(self, __context):
            """Fit TF-IDF ngay sau khi khởi tạo object."""
            corpus = [doc.page_content for doc in self.chunks]
            self._vectorizer = TfidfVectorizer(
                ngram_range=(1, 2),  # unigram + bigram
                min_df=1,
                max_df=0.95,
                sublinear_tf=True,   # dùng log(tf) thay tf thô
            )
            self._tfidf_matrix = self._vectorizer.fit_transform(corpus)

        def _get_relevant_documents(self, query: str) -> List[Document]:
            """Phương thức bắt buộc — được .invoke() của LangChain gọi nội bộ."""
            query_vec = self._vectorizer.transform([query])
            scores    = cosine_similarity(query_vec, self._tfidf_matrix).flatten()
            top_k_idx = np.argsort(scores)[::-1][:self.k]
            return [self.chunks[i] for i in top_k_idx]

        def invoke_with_score(self, query: str) -> List[tuple]:
            """Trả về (Document, score) — dùng cho get_retriever_docs(mode='dense')."""
            query_vec = self._vectorizer.transform([query])
            scores    = cosine_similarity(query_vec, self._tfidf_matrix).flatten()
            top_k_idx = np.argsort(scores)[::-1][:self.k]
            return [(self.chunks[i], round(float(scores[i]), 4)) for i in top_k_idx]

    print("⚙️  Đang fit TF-IDF trên toàn bộ corpus chunks...")
    dense_retriever = TfidfRetriever(chunks=chunks, k=5)
    print(f"✅ TF-IDF Retriever sẵn sàng!")
    print(f"   • Vocab size : {len(dense_retriever._vectorizer.vocabulary_):,} terms")
    print(f"   • Matrix     : {dense_retriever._tfidf_matrix.shape}  (chunks × terms)")
    print(f"   • Top-k      : {dense_retriever.k}")

    # ── 8. BM25 Retriever ─────────────────────────────────────────────────────
    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = 5

    # ── 9. RRF Hybrid + Re-ranking ────────────────────────────────────────────
    def _doc_key(doc):
        raw = f"{doc.metadata.get('source','')}|{doc.metadata.get('page','')}|{doc.page_content[:200]}"
        return hashlib.md5(raw.encode()).hexdigest()

    def rrf_hybrid(query, k=5, rrf_k=60):
        bm25_docs  = bm25_retriever.invoke(query)
        dense_docs = dense_retriever.invoke(query)   # gọi TfidfRetriever.invoke()

        scores = {}
        for rank, doc in enumerate(bm25_docs, start=1):
            key = _doc_key(doc)
            if key not in scores:
                scores[key] = {'score': 0.0, 'doc': doc}
            scores[key]['score'] += 1 / (rrf_k + rank)

        for rank, doc in enumerate(dense_docs, start=1):
            key = _doc_key(doc)
            if key not in scores:
                scores[key] = {'score': 0.0, 'doc': doc}
            scores[key]['score'] += 1 / (rrf_k + rank)

        ranked = sorted(scores.values(), key=lambda x: x['score'], reverse=True)
        return [(item['doc'], round(item['score'], 6)) for item in ranked[:k]]

    def rerank(query, doc_score_pairs, top_n=3):
        import re
        query_words = set(re.findall(r'\w+', query.lower()))
        if not query_words:
            return doc_score_pairs[:top_n]
        reranked = []
        for doc, rrf_score in doc_score_pairs:
            content_words = set(re.findall(r'\w+', doc.page_content.lower()))
            overlap = len(query_words & content_words) / (len(query_words) + 1e-9)
            final_score = round(0.7 * rrf_score * 100 + 0.3 * overlap, 6)
            reranked.append((doc, final_score))
        reranked.sort(key=lambda x: x[1], reverse=True)
        return reranked[:top_n]

    def get_retriever_docs(query, mode='hybrid', k=5):
        expanded = expand_query(query)
        if mode == 'bm25':
            docs = bm25_retriever.invoke(query)
            return [(d, round(1/(i+1), 4)) for i, d in enumerate(docs[:k])]
        elif mode == 'dense':
            results = dense_retriever.invoke_with_score(query)   # FIX: dùng method mới
            return [(d, round(float(s), 4)) for d, s in results]
        else:
            hybrid_results = rrf_hybrid(query, k=k+2)
            return rerank(query, hybrid_results, top_n=k)

    # ── 10. LLM ───────────────────────────────────────────────────────────────
    llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0)

    # ── 11. Prompt ────────────────────────────────────────────────────────────
    SYSTEM_PROMPT = """Bạn là **PTIT Knowledge Assistant** — một chuyên gia phân tích tài liệu học thuật của trường Học viện Công nghệ Bưu chính Viễn thông (PTIT).

## VAI TRÒ & NGUYÊN TẮC
- Bạn CHỈ trả lời dựa trên các đoạn tài liệu được cung cấp trong `[CONTEXT]`.
- Bạn KHÔNG suy diễn, KHÔNG thêm thông tin từ kiến thức nền ngoài tài liệu.
- Mọi thông tin đưa ra phải có trích dẫn số trang rõ ràng.

## QUY TẮC XỬ LÝ KHI KHÔNG TÌM THẤY THÔNG TIN
Nếu `[CONTEXT]` không chứa thông tin liên quan đến câu hỏi, hãy trả lời theo mẫu:
>  **Tài liệu không đề cập đến vấn đề này.**
> Nội dung được cung cấp không đủ để trả lời câu hỏi: *"{{question}}"*. Vui lòng tham khảo thêm tài liệu khác hoặc đặt câu hỏi khác.

## ĐỊNH DẠNG TRẢ LỜI (BẮT BUỘC)
Luôn trả lời theo cấu trúc Markdown sau:

###  Trả lời
[Nội dung trả lời chính — dùng bullet points nếu có nhiều ý, dùng bảng nếu cần so sánh]

###  Trích dẫn nguồn
- **[Tên file]**, Trang **[số trang]**: *[trích dẫn ngắn gọn câu liên quan từ tài liệu]*

---
*Thông tin được tổng hợp từ tài liệu nội bộ PTIT.*"""

    PROMPT = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "[CONTEXT]\n{context}\n\n[CÂU HỎI]\n{question}")
    ])

    def format_docs(doc_score_pairs):
        parts = []
        for d, score in doc_score_pairs:
            src  = d.metadata.get('source', 'unknown')
            page = d.metadata.get('page', '?')
            parts.append(f'[File: {src} | Trang: {page} | Score: {score}]\n{d.page_content}')
        return '\n\n---\n\n'.join(parts)

    def build_rag_chain(mode='hybrid'):
        return (
            {'context': RunnableLambda(lambda q: format_docs(get_retriever_docs(q, mode=mode))),
             'question': RunnablePassthrough()}
            | PROMPT | llm | StrOutputParser()
        )

    rag_chain = build_rag_chain('hybrid')

    print('\n🚀 HỆ THỐNG RAG SẴN SÀNG!')
    print(f'   • LLM        : gemini-2.5-flash')
    print(f'   • Embeddings : TF-IDF (scikit-learn) — chạy cục bộ')  # FIX: cập nhật log
    print(f'   • Chunking   : size=1000, overlap=100')
    print(f'   • Retrievers : BM25 | TF-IDF Dense | Hybrid RRF + Re-ranking')
    print(f'   • Chunks     : {len(chunks)}')
    all_chunks = list(chunks)

📂 Đang nạp PDF từ /content/ ...


100%|██████████| 1/1 [00:00<00:00,  3.28it/s]

✅ Đã nạp 46 trang từ 1 file.
🧹 Sau lọc trang 1–50: còn 46 trang.
✂️  Đã chia thành 46 chunks (chunk_size=1000, overlap=100).
✅ Tìm được 2 viết tắt:
   'rag' → 'retrieval augmented generation'
   'rr' → 'evaluating question answering reciprocal rank'
⚙️  Đang fit TF-IDF trên toàn bộ corpus chunks...
✅ TF-IDF Retriever sẵn sàng!
   • Vocab size : 1,478 terms
   • Matrix     : (46, 1478)  (chunks × terms)
   • Top-k      : 5

🚀 HỆ THỐNG RAG SẴN SÀNG!
   • LLM        : gemini-2.5-flash
   • Embeddings : TF-IDF (scikit-learn) — chạy cục bộ
   • Chunking   : size=1000, overlap=100
   • Retrievers : BM25 | TF-IDF Dense | Hybrid RRF + Re-ranking
   • Chunks     : 46


##  BƯỚC 3: Thước đo đánh giá (Evaluation Metrics)
Tính **Exact Match (EM)**, **F1 Score** và **Mean Reciprocal Rank (MRR)** theo giáo trình.

In [ ]:
import re, string, time
from collections import Counter

# ── Hàm chuẩn hoá văn bản ─────────────────────────────────────────────────────
def normalize(text):
    """Lowercase, xóa dấu câu và khoảng trắng thừa."""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

# ── Exact Match ───────────────────────────────────────────────────────────────
def exact_match(prediction, ground_truth):
    return int(normalize(prediction) == normalize(ground_truth))

# ── F1 Score ──────────────────────────────────────────────────────────────────
def f1_score(prediction, ground_truth):
    pred_tokens  = normalize(prediction).split()
    truth_tokens = normalize(ground_truth).split()
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_common = sum(common.values())
    if num_common == 0: return 0.0
    precision = num_common / len(pred_tokens)
    recall    = num_common / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)

# ── MRR ───────────────────────────────────────────────────────────────────────
def mean_reciprocal_rank(retrieved_lists, relevant_lists):
    rr_scores = []
    for retrieved, relevant in zip(retrieved_lists, relevant_lists):
        rr = 0.0
        for rank, doc in enumerate(retrieved, start=1):
            doc_text = normalize(doc.page_content if hasattr(doc, 'page_content') else str(doc))
            if any(normalize(r) in doc_text for r in relevant):
                rr = 1.0 / rank
                break
        rr_scores.append(rr)
    return sum(rr_scores) / len(rr_scores) if rr_scores else 0.0

# ── Hàm đánh giá (Đã thêm Delay để tránh lỗi 429) ──────────────────────────
def evaluate_retrievers(test_set, k=5):
    import pandas as pd
    results = {'mode': [], 'avg_em': [], 'avg_f1': [], 'mrr': []}

    for mode in ['bm25', 'dense', 'hybrid']:
        chain = build_rag_chain(mode)
        em_scores, f1_scores = [], []
        retrieved_lists, relevant_lists = [], []

        print(f'\n🔍 Đang đánh giá chế độ: {mode.upper()} ...')
        for i, item in enumerate(test_set):
            q = item['question']
            gt = item['ground_truth']
            keys = item.get('relevant_keywords', [gt])

            try:
                # Gọi LLM
                pred = chain.invoke(q)
                # Thêm delay ngắn giữa các câu hỏi để tránh rate limit (Gemini Free: 15 RPM)
                time.sleep(4)
            except Exception as e:
                pred = ''
                print(f'    Q{i+1} lỗi: {e}')
                time.sleep(10) # Đợi lâu hơn nếu gặp lỗi

            em_scores.append(exact_match(pred, gt))
            f1_scores.append(f1_score(pred, gt))

            # Lấy docs cho MRR
            doc_score_pairs = get_retriever_docs(q, mode=mode, k=k)
            retrieved_lists.append([d for d, _ in doc_score_pairs])
            relevant_lists.append(keys)

            print(f'  [{i+1}/{len(test_set)}] EM={em_scores[-1]} | F1={f1_scores[-1]:.2f}')

        # Nghỉ giữa các Mode
        print(f'✅ Xong mode {mode.upper()}. Nghỉ 10s chuyển sang mode tiếp theo...')
        time.sleep(10)

        mrr = mean_reciprocal_rank(retrieved_lists, relevant_lists)
        results['mode'].append(mode.upper())
        results['avg_em'].append(round(sum(em_scores)/len(em_scores), 4))
        results['avg_f1'].append(round(sum(f1_scores)/len(f1_scores), 4))
        results['mrr'].append(round(mrr, 4))

    return pd.DataFrame(results)

print('✅ Đã cập nhật hàm evaluate_retrievers với cơ chế delay chống lỗi Rate Limit.')

✅ Đã cập nhật hàm evaluate_retrievers với cơ chế delay chống lỗi Rate Limit.


## BƯỚC 4: Chạy đánh giá
Điền câu hỏi và đáp án mẫu vào `TEST_SET` bên dưới.

In [ ]:
import pandas as pd

# 1. Đọc file Excel
df_excel = pd.read_excel('Dataset_Demo_3_Cau_Chien_Thuat.xlsx')

# 2. Chuyển đổi dữ liệu thành định dạng danh sách các dictionary
# Lưu ý: Tên cột trong file Excel phải khớp với tên 'Câu hỏi' và 'Đáp án chuẩn'
TEST_SET = []
for index, row in df_excel.iterrows():
    TEST_SET.append({
        "question": row['Câu hỏi'],
        "ground_truth": row['Đáp án chuẩn']
    })

# 3. Kiểm tra thử 2 câu đầu tiên
print(f" Đã nạp thành công {len(TEST_SET)} câu hỏi vào TEST_SET.")
print(TEST_SET[:2])

 Đã nạp thành công 3 câu hỏi vào TEST_SET.
[{'question': 'Chỉ số MRR trong đánh giá hệ thống QA được tính toán dựa trên công thức nào?', 'ground_truth': 'Chỉ số MRR (Mean Reciprocal Rank) được tính bằng công thức: $MRR = \\frac{1}{|Q|} \\sum_{i=1}^{|Q|} \\frac{1}{rank_i}$, trong đó $|Q|$ là tổng số câu hỏi và $rank_i$ là vị trí của kết quả đúng đầu tiên.'}, {'question': 'Tại sao các mô hình ngôn ngữ lớn đôi khi lại trả lời sai sự thật một cách tự tin?', 'ground_truth': "Đây là hiện tượng 'Ảo giác' (Hallucinations). LLM có thể tạo ra nội dung trông có vẻ hợp lý nhưng thực tế là sai hoặc không có căn cứ khi đối mặt với thông tin lạ."}]


In [ ]:
import pandas as pd
import time

# --- 1. Nạp dữ liệu từ Excel ---
try:
    df_excel = pd.read_excel('Dataset_Demo_3_Cau_Chien_Thuat.xlsx')
    TEST_SET = [
        {"question": row['Câu hỏi'], "ground_truth": row['Đáp án chuẩn']}
        for _, row in df_excel.iterrows()
    ]
    print(f" Đã nạp {len(TEST_SET)} câu hỏi từ file Excel.")
except Exception as e:
    print(f" Lỗi nạp file: {e}. Hãy đảm bảo đã upload file .xlsx lên Colab.")

# --- 2. Chạy đánh giá với cơ chế chống lỗi API (Rate Limit) ---
print('\n' + '='*50)
print('📊 BẮT ĐẦU ĐÁNH GIÁ HỆ THỐNG')
print('='*50)

# Chạy đánh giá (Hàm này đã có delay nếu bạn đã sửa ở Bước 3,

df_results = evaluate_retrievers(TEST_SET, k=5)

# --- 3. Hiển thị bảng kết quả ---
print('\n' + '='*50)
print('📊 KẾT QUẢ SO SÁNH 3 PHƯƠNG THỨC TRUY XUẤT')
print('='*50)
print(df_results.to_string(index=False))

# Highlight phương thức tốt nhất
# Đã điều chỉnh tên cột để khớp với DataFrame từ hàm evaluate_retrievers.
best_f1 = df_results.loc[df_results['avg_f1'].idxmax(), 'mode']
print(f'\n🏆 Phương thức tối ưu nhất dựa trên F1 Score: {best_f1}')


 Đã nạp 3 câu hỏi từ file Excel.

📊 BẮT ĐẦU ĐÁNH GIÁ HỆ THỐNG

🔍 Đang đánh giá chế độ: BM25 ...
  [1/3] EM=0 | F1=0.24
  [2/3] EM=0 | F1=0.16
  [3/3] EM=0 | F1=0.25
✅ Xong mode BM25. Nghỉ 10s chuyển sang mode tiếp theo...

🔍 Đang đánh giá chế độ: DENSE ...
  [1/3] EM=0 | F1=0.24
  [2/3] EM=0 | F1=0.19
  [3/3] EM=0 | F1=0.30
✅ Xong mode DENSE. Nghỉ 10s chuyển sang mode tiếp theo...

🔍 Đang đánh giá chế độ: HYBRID ...
  [1/3] EM=0 | F1=0.24
  [2/3] EM=0 | F1=0.16
  [3/3] EM=0 | F1=0.30
✅ Xong mode HYBRID. Nghỉ 10s chuyển sang mode tiếp theo...

📊 KẾT QUẢ SO SÁNH 3 PHƯƠNG THỨC TRUY XUẤT
  mode  avg_em  avg_f1  mrr
  BM25     0.0  0.2148  0.0
 DENSE     0.0  0.2428  0.0
HYBRID     0.0  0.2303  0.0

🏆 Phương thức tối ưu nhất dựa trên F1 Score: DENSE


## BƯỚC 5: Giao diện Gradio nâng cao
Chọn phương thức truy xuất, xem câu trả lời + nguồn trích dẫn + điểm xếp hạng.

In [ ]:
# =============================================================================
# PATCH: Tính năng "Tải lên tài liệu mới" cho RAG_Gemini_v3.ipynb
#
# HƯỚNG DẪN TÍCH HỢP:
#   - PHẦN A → Thêm vào cuối ô "BƯỚC 2" (sau dòng print '🚀 HỆ THỐNG RAG...')
#   - PHẦN B → Thay thế toàn bộ ô "BƯỚC 5" (cell Gradio hiện tại)
# =============================================================================


# ─────────────────────────────────────────────────────────────────────────────
# PHẦN A — Thêm vào cuối ô BƯỚC 2
# Mục đích: khai báo all_chunks là master list, dùng chung với bm25_retriever
# ─────────────────────────────────────────────────────────────────────────────

# Đặt sau dòng: print(f'   • Chunks     : {len(chunks)}')
all_chunks = list(chunks)   # Master list — sẽ được extend mỗi khi upload


# ─────────────────────────────────────────────────────────────────────────────
# PHẦN B — Thay thế toàn bộ ô BƯỚC 5
# ─────────────────────────────────────────────────────────────────────────────

import gradio as gr

# ── CSS: giữ nguyên CSS gốc, thêm style cho upload panel ────────────────────
CUSTOM_CSS = """
.gradio-container { background: #0f1117 !important; }
.message.user {
    background: #1a3a5c !important; color: #e8f4fd !important;
    border-radius: 18px 18px 4px 18px !important;
    margin-left: auto !important; max-width: 75% !important;
    padding: 10px 15px !important; border: 1px solid #2a5a8c !important;
}
.message.bot {
    background: #1e2130 !important; color: #d4d8e8 !important;
    border-radius: 18px 18px 18px 4px !important;
    margin-right: auto !important; max-width: 85% !important;
    padding: 10px 15px !important; border: 1px solid #2e3250 !important;
}
.stat-card { background:#141720; border:1px solid rgba(0,200,255,0.25);
    border-radius:12px; padding:12px 16px; margin-bottom:10px;
    box-shadow:0 0 12px rgba(0,200,255,0.07); }
.stat-card .label { color:#7a8099; font-size:11px; text-transform:uppercase; letter-spacing:1px; }
.stat-card .value { color:#e2e8f0; font-size:14px; font-weight:600; margin-top:4px; }
.stat-card .value.green { color:#4ade80; } .stat-card .value.blue { color:#60a5fa; }
.source-tag { display:inline-block; background:rgba(0,200,255,0.1);
    border:1px solid rgba(0,200,255,0.3); border-radius:20px;
    padding:3px 12px; margin:3px 4px; font-size:12px; color:#7dd3fc; font-weight:500; }
.score-badge { display:inline-block; background:rgba(74,222,128,0.1);
    border:1px solid rgba(74,222,128,0.3); border-radius:8px;
    padding:2px 8px; margin-left:6px; font-size:11px; color:#4ade80; }
.sources-wrapper { padding:10px 4px 4px 4px; border-top:1px solid #2e3250; margin-top:6px; }
.sources-label { font-size:11px; color:#5a6080; text-transform:uppercase;
    letter-spacing:1px; margin-bottom:6px; }
.rounded-input textarea { border-radius:24px !important; padding:12px 20px !important;
    background:#1a1d2e !important; border:1px solid #2e3250 !important;
    color:#e2e8f0 !important; font-size:14px !important; }
.rounded-input textarea:focus { border-color:rgba(0,200,255,0.5) !important;
    box-shadow:0 0 0 3px rgba(0,200,255,0.1) !important; }
.send-btn { border-radius:24px !important; }
.chatbot-wrap .label-wrap { display:none !important; }
.chatbot-wrap { background:#0f1117 !important; border:1px solid #1e2130 !important;
    border-radius:16px !important; }
input[type=range] { accent-color:#00c8ff; }
h1.main-title { background:linear-gradient(90deg,#60a5fa,#00c8ff);
    -webkit-background-clip:text; -webkit-text-fill-color:transparent;
    font-size:26px !important; font-weight:700 !important; }

/* ── Upload panel ── */
.upload-status-ok  { color:#4ade80; font-size:13px; padding:6px 0; }
.upload-status-err { color:#f87171; font-size:13px; padding:6px 0; }
"""

MODE_LABELS = {'BM25 (Keyword)': 'bm25', 'Dense (Semantic)': 'dense', 'Hybrid RRF': 'hybrid'}

# ── Hàm build source HTML (giữ nguyên) ───────────────────────────────────────
def build_source_html(doc_score_pairs):
    seen, tags = set(), []
    for d, score in doc_score_pairs:
        key = (d.metadata.get('source', '?'), d.metadata.get('page', '?'))
        if key not in seen:
            seen.add(key)
            tags.append(
                f"<span class='source-tag'>📄 {key[0]} — Trang {key[1]}"
                f"<span class='score-badge'>score: {score}</span></span>"
            )
    if not tags:
        return ''
    return f"""
    <div class='sources-wrapper'>
        <div class='sources-label'>📚 Nguồn trích dẫn & Điểm xếp hạng</div>
        {''.join(tags)}
    </div>"""

# ── SIDEBAR_HTML: tĩnh, chunk count cập nhật qua gr.Markdown riêng ───────────
SIDEBAR_HTML = f"""
<div class='stat-card'><div class='label'>Trạng thái</div><div class='value green'>🟢 Sẵn sàng</div></div>
<div class='stat-card'><div class='label'>Mô hình LLM</div><div class='value blue'>gemini-2.5-flash</div></div>
<div class='stat-card'><div class='label'>Embeddings</div><div class='value blue'>gemini-embedding-001</div></div>
"""

def _chunk_count_md():
    """Trả về Markdown hiển thị tổng chunks — dùng để cập nhật động."""
    return f"<div class='stat-card'><div class='label'>Tổng số Chunks</div><div class='value'>{len(all_chunks):,} đoạn</div></div>"

# ── NEW: Hàm xử lý upload PDF ─────────────────────────────────────────────────
def handle_upload_pdfs(files):
    """
    Callback cho gr.File upload.

    Luồng xử lý:
      1. Đọc từng PDF bằng PyPDFLoader
      2. Chia nhỏ bằng RecursiveCharacterTextSplitter (size=1000, overlap=100)
      3. Thêm chunks mới vào vector_db (theo batch 80, delay 60s tránh 429)
      4. Rebuild bm25_retriever từ toàn bộ all_chunks (đảm bảo IDF chính xác)
      5. Trả về (status_message, chunk_count_html) để cập nhật UI
    """
    global vector_db, bm25_retriever, all_chunks, dense_retriever, ABBREV_MAP

    if not files:
        return "⚠️ Chưa chọn file nào.", _chunk_count_md()

    # ── Bước 1 & 2: Load + Chunk ─────────────────────────────────────────────
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    new_chunks  = []   # chunks từ lần upload này
    loaded_info = []   # log mỗi file
    errors      = []

    for file_obj in files:
        file_path = file_obj.name                   # Gradio trả về path tạm
        fname     = os.path.basename(file_path)

        try:
            # Load PDF
            loader = PyPDFLoader(file_path)
            pages  = loader.load()

            if not pages:
                errors.append(f"`{fname}`: không đọc được trang nào")
                continue

            # Làm sạch văn bản (khớp với pipeline Bước 2)
            clean_pages = []
            for doc in pages:
                lines = [ln.strip() for ln in doc.page_content.splitlines() if ln.strip()]
                doc.page_content      = '\n'.join(lines)
                doc.metadata['source'] = fname
                doc.metadata['page']   = doc.metadata.get('page', 0) + 1  # 1-based
                clean_pages.append(doc)

            # Chunk
            file_chunks = splitter.split_documents(clean_pages)
            new_chunks.extend(file_chunks)
            loaded_info.append(f"**{fname}** → {len(file_chunks)} chunks")

        except Exception as e:
            errors.append(f"`{fname}`: {e}")

    if not new_chunks:
        err_str = "\n".join(f"❌ {e}" for e in errors)
        return f"⚠️ Không trích xuất được chunk nào.\n{err_str}", _chunk_count_md()

    # ── Bước 3: Cập nhật Vector DB theo batch (tránh lỗi 429) ────────────────
    BATCH_SIZE    = 80
    total_batches = (len(new_chunks) + BATCH_SIZE - 1) // BATCH_SIZE

    for i in range(0, len(new_chunks), BATCH_SIZE):
        batch     = new_chunks[i : i + BATCH_SIZE]
        batch_num = i // BATCH_SIZE + 1
        vector_db.add_documents(batch)

        # Chờ 60s giữa các batch (Gemini embedding free-tier: ~100 req/min)
        if batch_num < total_batches:
            time.sleep(60)

    # ── Bước 4: Rebuild BM25 từ toàn bộ all_chunks ───────────────────────────
    # BM25 cần toàn bộ corpus để tính IDF chính xác → không thể incremental
    all_chunks.extend(new_chunks)
    bm25_retriever = BM25Retriever.from_documents(all_chunks)
    bm25_retriever.k = 5
    dense_retriever = TfidfRetriever(chunks=all_chunks, k=5)
    ABBREV_MAP      = build_abbrev_map(all_chunks)

    # ── Bước 5: Tạo thông báo kết quả ────────────────────────────────────────
    file_lines = "\n".join(f"  • {info}" for info in loaded_info)
    err_lines  = ("\n" + "\n".join(f"  ⚠️ {e}" for e in errors)) if errors else ""

    status = (
        f"✅ Đã nạp thêm **{len(new_chunks)} chunks** từ {len(loaded_info)} file:\n"
        f"{file_lines}"
        f"{err_lines}\n\n"
        f"📦 Tổng chunks trong hệ thống: **{len(all_chunks):,}**"
    )
    return status, _chunk_count_md()


# ── Streaming respond (giữ nguyên) ────────────────────────────────────────────
def respond_stream(message, chat_history, mode_label, top_k, temp):
    """Generator function cho streaming — yield từng token khi LLM sinh ra."""
    if not message.strip():
        yield '', chat_history, ''
        return

    mode = MODE_LABELS.get(mode_label, 'hybrid')
    llm.temperature = temp

    try:
        doc_scores  = get_retriever_docs(message, mode=mode, k=top_k)
        context_str = format_docs(doc_scores)
        source_html = build_source_html(doc_scores)

        prompt_val = PROMPT.format_messages(context=context_str, question=message)

        partial      = ''
        chat_history = chat_history + [(message, '')]
        for chunk in llm.stream(prompt_val):
            token    = chunk.content if hasattr(chunk, 'content') else str(chunk)
            partial += token
            chat_history[-1] = (message, partial)
            yield '', chat_history, source_html

    except Exception as e:
        chat_history = chat_history + [(message, f'❌ Lỗi: {str(e)}')]
        yield '', chat_history, ''


# ── Gradio Layout ─────────────────────────────────────────────────────────────
with gr.Blocks(
    css=CUSTOM_CSS,
    theme=gr.themes.Soft(primary_hue='blue', secondary_hue='slate',
                         neutral_hue='slate', font=gr.themes.GoogleFont('Inter')),
    title='PTIT AI - RAG System'
) as demo:

    gr.HTML("<h1 class='main-title'> Hệ Thống Hỏi Đáp Tài Liệu</h1>")

    with gr.Row():
        # ── Sidebar ──────────────────────────────────────────────────────────
        with gr.Column(scale=1, min_width=230):
            gr.Markdown('### ⚙️ Cấu hình')
            retriever_radio = gr.Radio(
                choices=list(MODE_LABELS.keys()),
                value='Hybrid RRF',
                label='🔍 Phương thức truy xuất',
            )
            top_k_slider = gr.Slider(minimum=1, maximum=10, value=5, step=1,
                                     label='Số đoạn trích dẫn (k)')
            temp_slider  = gr.Slider(minimum=0, maximum=1, value=0.0, step=0.1,
                                     label='Độ sáng tạo (Temperature)')

            gr.Markdown('---')

            # ── NEW: Khu vực tải lên tài liệu ───────────────────────────────
            gr.Markdown('### 📤 Tải lên tài liệu mới')

            upload_file = gr.File(
                file_count='multiple',       # cho phép chọn nhiều file cùng lúc
                file_types=['.pdf'],          # chỉ nhận PDF
                label='Chọn file PDF',
                interactive=True,
            )
            upload_btn = gr.Button(
                '📥 Nạp vào hệ thống',
                variant='secondary',
                size='sm',
            )
            # Hiển thị kết quả xử lý (Markdown để render bold/emoji)
            upload_status = gr.Markdown(
                value='',
                label='Trạng thái upload',
                visible=True,
            )

            gr.Markdown('---')

            # Stat cards tĩnh + chunk count động
            gr.HTML(SIDEBAR_HTML)
            chunk_count_display = gr.HTML(value=_chunk_count_md())  # cập nhật sau mỗi upload

            clear = gr.Button('🗑️ Xóa lịch sử', variant='stop')

        # ── Chat area (giữ nguyên) ────────────────────────────────────────────
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                show_label=False, height=480,
                show_copy_button=True, render_markdown=True,
                latex_delimiters=[
                    {'left': '$$', 'right': '$$', 'display': True},
                    {'left': '$',  'right': '$',  'display': False},
                ],
                elem_classes=['chatbot-wrap']
            )
            sources_display = gr.HTML('', elem_classes=['sources-area'])

            with gr.Row():
                msg = gr.Textbox(
                    placeholder='Nhập câu hỏi về tài liệu của bạn...',
                    show_label=False, scale=9, container=False,
                    elem_classes=['rounded-input']
                )
                submit = gr.Button('Gửi ➤', variant='primary', scale=1,
                                   elem_classes=['send-btn'])

    # ── Event wiring ──────────────────────────────────────────────────────────
    chat_inputs  = [msg, chatbot, retriever_radio, top_k_slider, temp_slider]
    chat_outputs = [msg, chatbot, sources_display]

    msg.submit(respond_stream, chat_inputs, chat_outputs)
    submit.click(respond_stream, chat_inputs, chat_outputs)
    clear.click(lambda: ([], ''), None, [chatbot, sources_display])

    # Upload: nút click → handle_upload_pdfs → cập nhật status + chunk count
    upload_btn.click(
        fn=handle_upload_pdfs,
        inputs=[upload_file],
        outputs=[upload_status, chunk_count_display],
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8223e015addfbc0b0b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

CUSTOM_CSS = """
.gradio-container { background: #0f1117 !important; }
.message.user {
    background: #1a3a5c !important; color: #e8f4fd !important;
    border-radius: 18px 18px 4px 18px !important;
    margin-left: auto !important; max-width: 75% !important;
    padding: 10px 15px !important; border: 1px solid #2a5a8c !important;
}
.message.bot {
    background: #1e2130 !important; color: #d4d8e8 !important;
    border-radius: 18px 18px 18px 4px !important;
    margin-right: auto !important; max-width: 85% !important;
    padding: 10px 15px !important; border: 1px solid #2e3250 !important;
}
.stat-card { background:#141720; border:1px solid rgba(0,200,255,0.25);
    border-radius:12px; padding:12px 16px; margin-bottom:10px;
    box-shadow:0 0 12px rgba(0,200,255,0.07); }
.stat-card .label { color:#7a8099; font-size:11px; text-transform:uppercase; letter-spacing:1px; }
.stat-card .value { color:#e2e8f0; font-size:14px; font-weight:600; margin-top:4px; }
.stat-card .value.green { color:#4ade80; } .stat-card .value.blue { color:#60a5fa; }
.source-tag { display:inline-block; background:rgba(0,200,255,0.1);
    border:1px solid rgba(0,200,255,0.3); border-radius:20px;
    padding:3px 12px; margin:3px 4px; font-size:12px; color:#7dd3fc; font-weight:500; }
.score-badge { display:inline-block; background:rgba(74,222,128,0.1);
    border:1px solid rgba(74,222,128,0.3); border-radius:8px;
    padding:2px 8px; margin-left:6px; font-size:11px; color:#4ade80; }
.sources-wrapper { padding:10px 4px 4px 4px; border-top:1px solid #2e3250; margin-top:6px; }
.sources-label { font-size:11px; color:#5a6080; text-transform:uppercase;
    letter-spacing:1px; margin-bottom:6px; }
.rounded-input textarea { border-radius:24px !important; padding:12px 20px !important;
    background:#1a1d2e !important; border:1px solid #2e3250 !important;
    color:#e2e8f0 !important; font-size:14px !important; }
.rounded-input textarea:focus { border-color:rgba(0,200,255,0.5) !important;
    box-shadow:0 0 0 3px rgba(0,200,255,0.1) !important; }
.send-btn { border-radius:24px !important; }
.chatbot-wrap .label-wrap { display:none !important; }
.chatbot-wrap { background:#0f1117 !important; border:1px solid #1e2130 !important;
    border-radius:16px !important; }
input[type=range] { accent-color:#00c8ff; }
h1.main-title { background:linear-gradient(90deg,#60a5fa,#00c8ff);
    -webkit-background-clip:text; -webkit-text-fill-color:transparent;
    font-size:26px !important; font-weight:700 !important; }
"""

MODE_LABELS = {'BM25 (Keyword)': 'bm25', 'Dense (Semantic)': 'dense', 'Hybrid RRF': 'hybrid'}

def build_source_html(doc_score_pairs):
    seen, tags = set(), []
    for d, score in doc_score_pairs:
        key = (d.metadata.get('source','?'), d.metadata.get('page','?'))
        if key not in seen:
            seen.add(key)
            tags.append(
                f"<span class='source-tag'>📄 {key[0]} — Trang {key[1]}"
                f"<span class='score-badge'>score: {score}</span></span>"
            )
    if not tags:
        return ''
    return f"""
    <div class='sources-wrapper'>
        <div class='sources-label'>📚 Nguồn trích dẫn & Điểm xếp hạng</div>
        {''.join(tags)}
    </div>"""

SIDEBAR_HTML = f"""
<div class='stat-card'><div class='label'>Trạng thái</div><div class='value green'>🟢 Sẵn sàng</div></div>
<div class='stat-card'><div class='label'>Mô hình LLM</div><div class='value blue'>gemini-2.5-flash</div></div>
<div class='stat-card'><div class='label'>Embeddings</div><div class='value blue'>gemini-embedding-001</div></div>
<div class='stat-card'><div class='label'>Tổng số Chunks</div><div class='value'>{len(chunks):,} đoạn</div></div>
"""

# ── Streaming respond ─────────────────────────────────────────────────────────
def respond_stream(message, chat_history, mode_label, top_k, temp):
    """Generator function cho streaming — yield từng token khi LLM sinh ra."""
    if not message.strip():
        yield '', chat_history, ''
        return

    mode = MODE_LABELS.get(mode_label, 'hybrid')
    llm.temperature = temp

    try:
        # Lấy docs và build context (không stream phần này)
        doc_scores  = get_retriever_docs(message, mode=mode, k=top_k)
        context_str = format_docs(doc_scores)
        source_html = build_source_html(doc_scores)

        # Build prompt message
        prompt_val = PROMPT.format_messages(context=context_str, question=message)

        # Stream từng token từ LLM
        partial = ''
        chat_history = chat_history + [(message, '')]
        for chunk in llm.stream(prompt_val):
            token = chunk.content if hasattr(chunk, 'content') else str(chunk)
            partial += token
            chat_history[-1] = (message, partial)
            yield '', chat_history, source_html   # cập nhật realtime

    except Exception as e:
        chat_history = chat_history + [(message, f'❌ Lỗi: {str(e)}')]
        yield '', chat_history, ''

with gr.Blocks(
    css=CUSTOM_CSS,
    theme=gr.themes.Soft(primary_hue='blue', secondary_hue='slate',
                         neutral_hue='slate', font=gr.themes.GoogleFont('Inter')),
    title='PTIT AI - RAG System'
) as demo:

    gr.HTML("<h1 class='main-title'>🤖 Hệ Thống Hỏi Đáp Tài Liệu</h1>")

    with gr.Row():
        with gr.Column(scale=1, min_width=230):
            gr.Markdown('### ⚙️ Cấu hình')
            retriever_radio = gr.Radio(
                choices=list(MODE_LABELS.keys()),
                value='Hybrid RRF',
                label='🔍 Phương thức truy xuất',
            )
            top_k_slider = gr.Slider(minimum=1, maximum=10, value=5, step=1,
                                     label='Số đoạn trích dẫn (k)')
            temp_slider  = gr.Slider(minimum=0, maximum=1, value=0.0, step=0.1,
                                     label='Độ sáng tạo (Temperature)')
            gr.Markdown('---')
            gr.HTML(SIDEBAR_HTML)
            clear = gr.Button('🗑️ Xóa lịch sử', variant='stop')

        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                show_label=False, height=480,
                show_copy_button=True, render_markdown=True,
                latex_delimiters=[
                    {'left': '$$', 'right': '$$', 'display': True},
                    {'left': '$',  'right': '$',  'display': False},
                ],
                elem_classes=['chatbot-wrap']
            )
            sources_display = gr.HTML('', elem_classes=['sources-area'])

            with gr.Row():
                msg = gr.Textbox(
                    placeholder='Nhập câu hỏi về tài liệu của bạn...',
                    show_label=False, scale=9, container=False,
                    elem_classes=['rounded-input']
                )
                submit = gr.Button('Gửi ➤', variant='primary', scale=1,
                                   elem_classes=['send-btn'])

    inputs  = [msg, chatbot, retriever_radio, top_k_slider, temp_slider]
    outputs = [msg, chatbot, sources_display]

    # stream=True bật chế độ streaming cho generator
    msg.submit(respond_stream, inputs, outputs)
    submit.click(respond_stream, inputs, outputs)
    clear.click(lambda: ([], ''), None, [chatbot, sources_display])

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8afaea8bb480172531.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 🔬 BƯỚC 6 (Tuỳ chọn): So sánh BM25 vs Dense vs Hybrid qua terminal

In [ ]:
def compare_retrievers(query, k=3):
    print(f'🔍 Query: "{query}"')
    print('=' * 70)
    for mode, label in [('bm25','📊 BM25'), ('dense','🧠 Dense'), ('hybrid','⚡ Hybrid RRF')]:
        print(f'\n{label} — Top {k} kết quả:')
        for rank, (doc, score) in enumerate(get_retriever_docs(query, mode=mode, k=k), 1):
            src  = doc.metadata.get('source','?')
            page = doc.metadata.get('page','?')
            print(f'  [{rank}] {src} - Trang {page} | score={score}')
            print(f'      {doc.page_content[:120].strip()}...')

test_query = input('Nhập câu hỏi để so sánh: ')
if test_query:
    compare_retrievers(test_query)


Nhập câu hỏi để so sánh: Cách tính điểm MRR (Mean Reciprocal Rank) cho một hệ thống QA?
🔍 Query: "Cách tính điểm MRR (Mean Reciprocal Rank) cho một hệ thống QA?"

📊 BM25 — Top 3 kết quả:
  [1] Chapter 8_ Question Answering.pdf - Trang 46 | score=1.0
      Evaluating Question Answering
3 queries and their ranks of the first relevant documents are:
• Query 1: First relevant r...
  [2] Chapter 8_ Question Answering.pdf - Trang 44 | score=0.5
      Evaluating Question Answering
QA systems give multiple ranked answers, we evaluated by using
mean reciprocal rank, or MR...
  [3] Chapter 8_ Question Answering.pdf - Trang 45 | score=0.3333
      Evaluating Question Answering
Reciprocal Rank (RR):
• For each query, the RR is calculated. If the first relevant result...

🧠 Dense — Top 3 kết quả:
  [1] Chapter 8_ Question Answering.pdf - Trang 46 | score=0.4198
      Evaluating Question Answering
3 queries and their ranks of the first relevant documents are:
• Query 1: First relevant r...
  [2] Cha